# Femmestral — Ministral 3B Fine-Tune

Fine-tunes Ministral 3B on women's health misinformation using Unsloth + LoRA.  
Logs training to W&B Models, traces inference with Weave, pushes adapter to HuggingFace.

**Before running:** Add these to Colab Secrets (left sidebar → key icon):
- `WANDB_API_KEY`
- `HF_TOKEN`
- `GITHUB_TOKEN` (classic PAT with repo read access)

In [ ]:
# Cell 1 — Install dependencies
%%capture
!pip install unsloth
!pip install trl transformers accelerate peft bitsandbytes
!pip install wandb weave huggingface_hub datasets

In [ ]:
# Cell 2 — Authenticate
import os
from google.colab import userdata

os.environ["WANDB_API_KEY"]  = userdata.get("WANDB_API_KEY")
os.environ["HF_TOKEN"]       = userdata.get("HF_TOKEN")
os.environ["WANDB_PROJECT"]  = "femmestral"

import wandb
import weave

run = wandb.init(
    project="femmestral",
    job_type="fine-tune",
    config={
        "base_model":      "mistralai/Ministral-7B-Instruct-2410",
        "lora_r":          16,
        "lora_alpha":      16,
        "training_steps":  100,
        "learning_rate":   1e-4,
        "batch_size":      2,
        "grad_accum":      4,
        "max_seq_length":  2048,
        "train_examples":  240,
        "eval_examples":   60,
        "task":            "women_health_misinformation",
    }
)
weave.init("femmestral")
print("W&B and Weave initialised.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: elenaajayi (elenaajayi-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Initializing weave.
weave: Logged in as Weights & Biases user: elenaajayi.
weave: View Weave data at https://wandb.ai/elenaajayi-n-a/femmestral/weave


W&B and Weave initialised.


In [ ]:
  # Cell 3 — Clone repo and load data
  import subprocess, json

  # Remove existing directory if it exists from a previous attempt
  subprocess.run(["rm", "-rf", "femmestral"], check=False)

  subprocess.run(
      ["git", "clone", "--branch", "elena_mistral_hack",
       "https://github.com/elenaajayi/femmestral.git"],
      check=True
  )

  def load_jsonl(path):
      with open(path) as f:
          return [json.loads(line) for line in f if line.strip()]

  train_data = load_jsonl("femmestral/data/train.jsonl")
  eval_data  = load_jsonl("femmestral/data/eval.jsonl")
  print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")



Train: 240 | Eval: 60


In [ ]:
# Cell 4 — Load Ministral 7B in 4-bit via Unsloth
from unsloth import FastLanguageModel
import torch

max_seq_length = run.config["max_seq_length"]

model, tokenizer = FastLanguageModel.from_pretrained(
      model_name    = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit" ,
      max_seq_length= max_seq_length,
      dtype         = None,
      load_in_4bit  = True,
)

print("Model loaded.")

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded.


In [ ]:
# Cell 5 — Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r                      = run.config["lora_r"],
    target_modules         = ["q_proj", "k_proj", "v_proj", "o_proj",
                               "gate_proj", "up_proj", "down_proj"],
    lora_alpha             = run.config["lora_alpha"],
    lora_dropout           = 0,
    bias                   = "none",
    use_gradient_checkpointing = "unsloth",
    random_state           = 42,
)
print("LoRA adapters applied.")

Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA adapters applied.


In [ ]:
# Cell 6 — Prepare datasets
from datasets import Dataset

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

train_dataset = Dataset.from_list(train_data).map(format_example)
eval_dataset  = Dataset.from_list(eval_data).map(format_example)
print(f"Datasets formatted. Example:\n{train_dataset[0]['text'][:300]}")

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Datasets formatted. Example:
<s>[INST] Deodorant with aluminium causes breast cancer — chemicals go directly from armpit to breast tissue!! Switch to natural deodorant NOW!![/INST] **Verdict: False**
**Confidence: High** | **Evidence: B** | **Severity: High**

The National Cancer Institute and Cancer Research UK have concluded 


In [ ]:
# Cell 7 — Train
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = train_dataset,
    eval_dataset      = eval_dataset,
    dataset_text_field= "text",
    max_seq_length    = max_seq_length,
    dataset_num_proc  = 2,
    args = TrainingArguments(
        per_device_train_batch_size = run.config["batch_size"],
        gradient_accumulation_steps = run.config["grad_accum"],
        warmup_steps                = 5,
        max_steps                   = run.config["training_steps"],
        learning_rate               = run.config["learning_rate"],
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 1,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 42,
        output_dir                  = "outputs",
        report_to                   = "wandb",
        eval_strategy               = "steps",
        eval_steps                  = 20,
        save_strategy               = "steps",
        save_steps                  = 20,
        load_best_model_at_end      = True,
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/240 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/60 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 240 | Num Epochs = 4 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Step,Training Loss,Validation Loss
20,1.116500,1.126291
40,0.845100,0.992846
60,0.806100,0.925821
80,0.588900,0.964149
100,0.535800,0.940247


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


eval/loss,█▃▁▂▂
eval/runtime,█▂▂▁▁
eval/samples_per_second,▁▇▇██
eval/steps_per_second,▁▇▇██
train/epoch,▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,█▅▃ ▄▁▁▁▁▁▁▁▁▁▂▁▁▂▂▂▂▂▂▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂
train/learning_rate,▄▇███████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▂▂▂▂▁▁▁▁▁
train/loss,██▅▅▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▂▁▁▁▁▁▁▁▁▁
eval/loss,0.94025
eval/runtime,14.326


TrainOutput(global_step=100, training_loss=0.9204305186867714, metrics={'train_runtime': 505.8632, 'train_samples_per_second': 1.581, 'train_steps_per_second': 0.198, 'total_flos': 6872696061935616.0, 'train_loss': 0.9204305186867714, 'epoch': 3.3333333333333335})

In [ ]:
# Cell 8 — Push LoRA adapter to HuggingFace
HF_REPO = "elenaajayi/femmestral-mistral-7b"

model.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
print(f"Pushed to https://huggingface.co/{HF_REPO}")

Saved model to https://huggingface.co/elenaajayi/femmestral-mistral-7b
Pushed to https://huggingface.co/elenaajayi/femmestral-mistral-7b


In [ ]:
  # Cell 9 — Log W&B artifact
  run = wandb.init(project="femmestral", resume="must", id="ogejyi0g")

  artifact = wandb.Artifact(
      name        = "mistral-7b-femmestral",
      type        = "model",
      description = "Mistral 7B LoRA adapter fine-tuned on 240 women's health misinformation examples.",
      metadata    = {
          "base_model":      "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
          "lora_r":          16,
          "training_steps":  100,
          "learning_rate":   1e-4,
          "train_examples":  240,
          "eval_examples":   60,
          "task":            "women_health_misinformation",
          "hf_repo":         HF_REPO,
          "access":          f"PeftModel.from_pretrained(base, '{HF_REPO}')",
      }
  )

  artifact.add_file("femmestral/data/train.jsonl", name="train.jsonl")
  artifact.add_file("femmestral/data/eval.jsonl",  name="eval.jsonl")

  with open("/tmp/model_ref.txt", "w") as f:
      f.write(f"LoRA adapter: https://huggingface.co/{HF_REPO}\n\n")
      f.write("Load:\n")
      f.write("  from peft import PeftModel\n")
      f.write("  from transformers import AutoModelForCausalLM\n")
      f.write(f"  base = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-7B-Instruct-v0.3')\n")
      f.write(f"  model = PeftModel.from_pretrained(base, '{HF_REPO}')\n")

  artifact.add_file("/tmp/model_ref.txt", name="model_ref.txt")
  run.log_artifact(artifact)
  print("W&B artifact logged.")

wandb: Initializing weave.


Output()

weave: Logged in as Weights & Biases user: elenaajayi.
weave: View Weave data at https://wandb.ai/elenaajayi-n-a/femmestral/weave


W&B artifact logged.


In [ ]:
# Cell 10 — Sample inference with Weave tracing
FastLanguageModel.for_inference(model)

@weave.op()
def classify_claim(claim: str) -> str:
    messages = [{"role": "user", "content": claim}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids      = inputs,
        max_new_tokens = 600,
        temperature    = 0.3,
        do_sample      = True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

test_claims = [
    "HPV vaccine causes infertility!! Share with all women!!!",
    "Eating papaya causes miscarriage. Warn all pregnant women!!",
    "Birth control pills cause permanent infertility after 2 years of use.",
    "Mammograms cause breast cancer from the radiation.",
    "Women cannot get pregnant while breastfeeding — it is 100% safe.",
]

results = []
for claim in test_claims:
    response = classify_claim(claim)
    results.append({"claim": claim, "response": response})
    print(f"\nClaim: {claim}\nResponse: {response[:300]}...\n{'='*60}")

run.log({"sample_outputs": wandb.Table(
    columns=["claim", "response"],
    data=[[r["claim"], r["response"]] for r in results]
)})

wandb.finish()
print("Done.")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca695-2864-7c65-ab47-94e5edd10e1f
weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca695-53a5-7dba-b81f-4c2abe02226c



Claim: HPV vaccine causes infertility!! Share with all women!!!
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Multiple large studies have found no evidence that the HPV vaccine affects fertility. The vaccine protects against cervical cancer and other HPV-related cancers. It is safe and recommended for adolescents.

**Source:** C...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca695-7ad1-7b90-9c34-635ade040ab0



Claim: Eating papaya causes miscarriage. Warn all pregnant women!!
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: Medium**

Papaya is safe during pregnancy. The myth that papaya causes miscarriage has no scientific basis. Green, unripe papaya contains latex that may stimulate contractions in large amounts, but ripe papaya is safe.

**Source...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca695-9b5a-7b69-aeb5-75be304690b3



Claim: Birth control pills cause permanent infertility after 2 years of use.
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: Medium**

Birth control pills do not cause permanent infertility. Fertility usually returns within 3–6 months of stopping. The pill affects ovulation temporarily while being taken.

**Source:** NIH Office on Women's Health — Con...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca695-c46f-71fd-8c89-35e103e44b92



Claim: Mammograms cause breast cancer from the radiation.
Response: **Verdict: False**
**Confidence: High** | **Evidence: B** | **Severity: High**

Mammograms use very low doses of radiation. The risk from a single mammogram is extremely small. The benefit of early cancer detection far outweighs this minuscule risk.

**Source:** NIH — Mammography and Radiation Risk
...

Claim: Women cannot get pregnant while breastfeeding — it is 100% safe.
Response: **Verdict: Misleading**
**Confidence: Medium** | **Evidence: A** | **Severity: Medium**

The Lactational Amenorrhoea Method (LAM) is a form of natural family planning that can work for some women, but it is not 100% effective and is not a reliable method of contraception. Breastfeeding does delay ov...


eval/loss,0.94025
eval/runtime,14.326
eval/samples_per_second,4.188
eval/steps_per_second,2.094
total_flos,6872696061935616
train/epoch,3.33333
train/global_step,100
train/grad_norm,1.6414
train/learning_rate,0.0
train/loss,0.5358
+4,...


Done.


In [ ]:
# %%capture
!pip install wandb weave

import os
import wandb
import weave
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["HF_TOKEN"]      = userdata.get("HF_TOKEN")

run2 = wandb.init(
    project="femmestral",
    job_type="fine-tune",
    config={
        "base_model":      "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
        "lora_r":          16,
        "lora_alpha":      16,
        "training_steps":  80,
        "learning_rate":   5e-5,
        "batch_size":      2,
        "grad_accum":      4,
        "max_seq_length":  2048,
        "train_examples":  240,
        "eval_examples":   60,
        "task":            "women_health_misinformation",
        "note":            "Run 2 — lower LR to reduce overfitting",
    }
)
weave.init("femmestral")
print("W&B run initialised.")

wandb: Initializing weave.


Output()

weave: Logged in as Weights & Biases user: elenaajayi.
weave: View Weave data at https://wandb.ai/elenaajayi-n-a/femmestral/weave


W&B run initialised.


In [ ]:

  # Run 2 — Cell B: Install + Load model
  # %%capture
  !pip install unsloth trl transformers accelerate peft bitsandbytes datasets

  from unsloth import FastLanguageModel
  from unsloth import is_bfloat16_supported
  import torch

  model2, tokenizer2 = FastLanguageModel.from_pretrained(
      model_name    = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
      max_seq_length= 2048,
      dtype         = None,
      load_in_4bit  = True,
  )

  model2 = FastLanguageModel.get_peft_model(
      model2,
      r                          = 16,
      target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                     "gate_proj", "up_proj", "down_proj"],
      lora_alpha                 = 16,
      lora_dropout               = 0,
      bias                       = "none",
      use_gradient_checkpointing = "unsloth",
      random_state               = 42,
  )
  print("Model 2 loaded and LoRA applied.")

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model 2 loaded and LoRA applied.


In [ ]:
  # Run 2 — Cell C: Load data + Train
  import subprocess, json
  from datasets import Dataset
  from trl import SFTTrainer
  from transformers import TrainingArguments

  # Reload data (in case of disconnect)
  subprocess.run(["rm", "-rf", "femmestral"], check=False)
  subprocess.run(["git", "clone", "--branch", "elena_mistral_hack",
                  "https://github.com/elenaajayi/femmestral.git"], check=True)

  def load_jsonl(path):
      with open(path) as f:
          return [json.loads(line) for line in f if line.strip()]

  train_data = load_jsonl("femmestral/data/train.jsonl")
  eval_data  = load_jsonl("femmestral/data/eval.jsonl")

  def format_example(example):
      text = tokenizer2.apply_chat_template(
          example["messages"], tokenize=False, add_generation_prompt=False
      )
      return {"text": text}

  train_dataset = Dataset.from_list(train_data).map(format_example)
  eval_dataset  = Dataset.from_list(eval_data).map(format_example)

  trainer2 = SFTTrainer(
      model             = model2,
      tokenizer         = tokenizer2,
      train_dataset     = train_dataset,
      eval_dataset      = eval_dataset,
      dataset_text_field= "text",
      max_seq_length    = 2048,
      dataset_num_proc  = 2,
      args = TrainingArguments(
          per_device_train_batch_size = 2,
          gradient_accumulation_steps = 4,
          warmup_steps                = 5,
          max_steps                   = 80,
          learning_rate               = 5e-5,
          fp16                        = not is_bfloat16_supported(),
          bf16                        = is_bfloat16_supported(),
          logging_steps               = 1,
          optim                       = "adamw_8bit",
          weight_decay                = 0.01,
          lr_scheduler_type           = "cosine",
          seed                        = 42,
          output_dir                  = "outputs2",
          report_to                   = "wandb",
          eval_strategy               = "steps",
          eval_steps                  = 20,
          save_strategy               = "steps",
          save_steps                  = 20,
          load_best_model_at_end      = True,
      ),
  )
  trainer2.train()

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/240 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/60 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 240 | Num Epochs = 3 | Total steps = 80
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Step,Training Loss,Validation Loss
20,1.332200,1.315043
40,1.022900,1.082266
60,1.001100,1.033853
80,0.891900,1.026828


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


eval/loss,█▂▁▁
eval/runtime,█▁▁▁
eval/samples_per_second,▁▇██
eval/steps_per_second,▁▇██
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
train/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train/grad_norm,█▄▄▂ ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▁▁▂▁▂▂▁▁▁▁
train/learning_rate,▁▂▄▇██████▇▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁
train/loss,███▅▄▄▃▃▃▃▂▃▂▃▂▂▂▂▂▂▂▂▁▂▁▁▂▁▂▂▁▁▁▁▁▁▁▁▁▁
eval/loss,1.02683
eval/runtime,14.2932


TrainOutput(global_step=80, training_loss=1.23151036798954, metrics={'train_runtime': 445.2653, 'train_samples_per_second': 1.437, 'train_steps_per_second': 0.18, 'total_flos': 5494842306674688.0, 'train_loss': 1.23151036798954, 'epoch': 2.6666666666666665})

In [16]:
  # Run 2 — Cell D: Push to HuggingFace + log artifact
  import wandb
  import os
  from google.colab import userdata

  os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

  HF_REPO2 = "elenaajayi/femmestral-mistral-7b-v2"

  # Resume run 2
  run2 = wandb.init(project="femmestral", resume="must", id="971dkrfv")

  model2.push_to_hub(HF_REPO2, token=os.environ["HF_TOKEN"])
  tokenizer2.push_to_hub(HF_REPO2, token=os.environ["HF_TOKEN"])
  print(f"Pushed to https://huggingface.co/{HF_REPO2}")

  artifact2 = wandb.Artifact(
      name        = "mistral-7b-femmestral-v2",
      type        = "model",
      description = "Run 2 — lower LR (5e-5), 80 steps, reduced overfitting.",
      metadata    = {
          "base_model":     "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
          "learning_rate":  5e-5,
          "training_steps": 80,
          "hf_repo":        HF_REPO2,
      }
  )
  artifact2.add_file("femmestral/data/train.jsonl", name="train.jsonl")
  artifact2.add_file("femmestral/data/eval.jsonl",  name="eval.jsonl")
  run2.log_artifact(artifact2)
  print("Artifact logged.")

wandb: Initializing weave.


Output()

weave: Logged in as Weights & Biases user: elenaajayi.
weave: View Weave data at https://wandb.ai/elenaajayi-n-a/femmestral/weave


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pvp5chue7/tokenizer.model: 100%|##########|  587kB /  587kB            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to https://huggingface.co/elenaajayi/femmestral-mistral-7b-v2
Artifact logged.


In [17]:

  # Run 2 — Cell E: Inference + Weave tracing + finish
  import wandb
  import weave
  from unsloth import FastLanguageModel

  FastLanguageModel.for_inference(model2)

  @weave.op()
  def classify_claim_v2(claim: str) -> str:
      messages = [{"role": "user", "content": claim}]
      inputs = tokenizer2.apply_chat_template(
          messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
      ).to("cuda")
      attention_mask = inputs.ne(tokenizer2.pad_token_id)
      outputs = model2.generate(
          input_ids      = inputs,
          attention_mask = attention_mask,
          max_new_tokens = 600,
          temperature    = 0.3,
          do_sample      = True,
      )
      return tokenizer2.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

  test_claims = [
      "HPV vaccine causes infertility!! Share with all women!!!",
      "Eating papaya causes miscarriage. Warn all pregnant women!!",
      "Birth control pills cause permanent infertility after 2 years of use.",
      "Mammograms cause breast cancer from the radiation.",
      "Women cannot get pregnant while breastfeeding — it is 100% safe.",
  ]

  results2 = []
  for claim in test_claims:
      response = classify_claim_v2(claim)
      results2.append({"claim": claim, "response": response})
      print(f"\nClaim: {claim}\nResponse: {response[:300]}...\n{'='*60}")

  run2.log({"sample_outputs": wandb.Table(
      columns=["claim", "response"],
      data=[[r["claim"], r["response"]] for r in results2]
  )})

  wandb.finish()
  print("Run 2 done.")

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca6c6-2549-7bbf-a34a-ae9fa1a76969
weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca6c6-57a9-72e7-91b9-7a5890132ef5



Claim: HPV vaccine causes infertility!! Share with all women!!!
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Multiple large studies have found no evidence that the HPV vaccine affects fertility. The vaccine protects against cervical cancer and other HPV-related cancers and diseases. It does not cause infertility.

**Source:** W...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca6c6-8253-7d5c-9a97-be4e7f6bf75c



Claim: Eating papaya causes miscarriage. Warn all pregnant women!!
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Papaya is safe during pregnancy. The unripe fruit contains latex that can stimulate uterine contractions in large amounts, but ripe papaya is safe and rich in vitamins A and C. This claim is not supported by evidence.

*...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca6c6-a96b-7523-821a-dd3f17ae47cc



Claim: Birth control pills cause permanent infertility after 2 years of use.
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Birth control pills do not cause permanent infertility. Fertility returns within 3–6 months of stopping. The pill does not affect the number or quality of eggs.

**Source:** WHO Medical Eligibility Criteria for Contracep...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca6c6-d327-7491-bb26-08ab8f00b762



Claim: Mammograms cause breast cancer from the radiation.
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Mammograms use a very low dose of radiation — far less than a single CT scan. The risk of radiation-induced cancer from mammography is extremely small. The benefits of early detection far outweigh this risk.

**Source:**...

Claim: Women cannot get pregnant while breastfeeding — it is 100% safe.
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: Medium**

Exclusive breastfeeding provides some protection against ovulation, but it is not foolproof. Many women resume periods within 6 months and can conceive while breastfeeding. The Lactational Amenorrhoea Method (LAM) is 9...


eval/loss,1.02683
eval/runtime,14.2932
eval/samples_per_second,4.198
eval/steps_per_second,2.099
total_flos,5494842306674688
train/epoch,2.66667
train/global_step,80
train/grad_norm,1.37586
train/learning_rate,0.0
train/loss,0.8919
+4,...


Run 2 done.


In [28]:
  import subprocess, sys, os

  subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                  "wandb", "weave", "requests", "python-dotenv",
                  "transformers", "peft", "accelerate"], capture_output=True)

  subprocess.run(["rm", "-rf", "femmestral"], check=False)
  subprocess.run(["git", "clone", "--branch", "elena_mistral_hack",
                  "https://github.com/elenaajayi/femmestral.git"], check=True)

  sys.path.insert(0, "femmestral")

  from google.colab import userdata
  import wandb, weave

  os.environ["NCBI_API_KEY"]  = userdata.get("NCBI_API_KEY")
  os.environ["HF_TOKEN"]      = userdata.get("HF_TOKEN")
  os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

  weave.init("femmestral")

  from src.rag.pubmed import search_pubmed
  from src.rag.semantic_scholar import search_semantic_scholar
  from src.reasoning.fact_checker import retrieve_evidence

  test_claims = [
      "HPV vaccine causes infertility!! Share with all women!!!",
      "Eating papaya causes miscarriage. Warn all pregnant women!!",
      "Birth control pills cause permanent infertility after 2 years of use.",
  ]

  for claim in test_claims:
      print(f"\nClaim: {claim}")
      evidence = retrieve_evidence(claim)
      print(f"Total evidence: {len(evidence)} papers")
      for e in evidence:
          print(f"  [{e['evidence_grade']}] {e['title'][:70]}...")
          print(f"  {e['url']}")


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca719-3ecd-7402-950a-3b60665aeced



Claim: HPV vaccine causes infertility!! Share with all women!!!


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca719-4913-73b0-8bb1-2c7ccc30f2ae


Total evidence: 2 papers
  [B] Attitudes towards human papillomavirus vaccination among African paren...
  https://pubmed.ncbi.nlm.nih.gov/27549328/
  [B] Overcoming barriers to HPV vaccination in Pakistan: a critical need fo...
  https://pubmed.ncbi.nlm.nih.gov/41497052/

Claim: Eating papaya causes miscarriage. Warn all pregnant women!!


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca719-4ce4-7b49-a0ca-d577dd8fa831


Total evidence: 0 papers

Claim: Birth control pills cause permanent infertility after 2 years of use.
Total evidence: 3 papers
  [A] Contraception and women's health....
  https://pubmed.ncbi.nlm.nih.gov/8324613/
  [B] Etiology, clinical features and prognosis in secondary amenorrhea....
  https://pubmed.ncbi.nlm.nih.gov/20414/
  [B] Association between self-reported questionnaire data on fertility and ...
  https://pubmed.ncbi.nlm.nih.gov/22563903/


In [33]:
# Reload both models after disconnect
import subprocess, sys, os

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "unsloth", "transformers", "peft", "accelerate",
                "bitsandbytes", "wandb", "weave", "requests"], capture_output=True)

subprocess.run(["rm", "-rf", "femmestral"], check=False)
subprocess.run(["git", "clone", "--branch", "elena_mistral_hack",
                "https://github.com/elenaajayi/femmestral.git"], check=True)

sys.path.insert(0, "femmestral")

from google.colab import userdata
import wandb, weave, os

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["HF_TOKEN"]      = userdata.get("HF_TOKEN")
os.environ["NCBI_API_KEY"]  = userdata.get("NCBI_API_KEY")

weave.init("femmestral")

from unsloth import FastLanguageModel

# Run 1 — best eval loss (0.940)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "elenaajayi/femmestral-mistral-7b",
    max_seq_length= 2048,
    dtype         = None,
    load_in_4bit  = True,
    token         = os.environ["HF_TOKEN"],
)

# Run 2 — more stable training
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name    = "elenaajayi/femmestral-mistral-7b-v2",
    max_seq_length= 2048,
    dtype         = None,
    load_in_4bit  = True,
    token         = os.environ["HF_TOKEN"],
)

FastLanguageModel.for_inference(model)
FastLanguageModel.for_inference(model2)

print("Both models loaded and ready.")

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Both models loaded and ready.


In [35]:
# Full RAG Pipeline — retrieval + fine-tuned model inference
# We run both Run 1 and Run 2 models for comparison:
# - model  (Run 1): lr=1e-4, 100 steps, eval loss 0.940 — lower loss, best checkpoint
# - model2 (Run 2): lr=5e-5, 80 steps,  eval loss 1.027 — more stable, less overfitting
# Run 1 is the better model by eval loss. Run 2 shows the optimization attempt.
# Both are logged to Weave for comparison in the W&B Report.

from unsloth import FastLanguageModel
from src.reasoning.fact_checker import retrieve_evidence, _build_prompt, SYSTEM_PROMPT

FastLanguageModel.for_inference(model)
FastLanguageModel.for_inference(model2)

test_claims = [
    "HPV vaccine causes infertility!! Share with all women!!!",
    "Eating papaya causes miscarriage. Warn all pregnant women!!",
    "Birth control pills cause permanent infertility after 2 years of use.",
]

def run_pipeline(claim: str, mdl, tok, run_name: str) -> dict:
    evidence = retrieve_evidence(claim)
    prompt   = _build_prompt(claim, evidence)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    inputs = tok.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = mdl.generate(
        input_ids      = inputs,
        max_new_tokens = 600,
        temperature    = 0.3,
        do_sample      = True,
    )
    response = tok.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return {
        "run":      run_name,
        "claim":    claim,
        "evidence": [e["title"] for e in evidence],
        "response": response,
    }

@weave.op()
def full_pipeline_run1(claim: str) -> dict:
    # Run 1 — best eval loss (0.940), lr=1e-4, 100 steps
    return run_pipeline(claim, model, tokenizer, "run1_lr1e-4")

@weave.op()
def full_pipeline_run2(claim: str) -> dict:
    # Run 2 — more stable training, less overfitting, lr=5e-5, 80 steps
    return run_pipeline(claim, model2, tokenizer2, "run2_lr5e-5")

for claim in test_claims:
    print(f"\n{'='*60}\nClaim: {claim}\n")

    r1 = full_pipeline_run1(claim)
    print(f"[Run 1] Evidence: {len(r1['evidence'])} papers")
    print(f"[Run 1] Response:\n{r1['response']}\n")

    r2 = full_pipeline_run2(claim)
    print(f"[Run 2] Evidence: {len(r2['evidence'])} papers")
    print(f"[Run 2] Response:\n{r2['response']}")


Claim: HPV vaccine causes infertility!! Share with all women!!!



weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca722-4574-7715-8206-8cd7cb1a7aea
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[Run 1] Evidence: 2 papers
[Run 1] Response:
**Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

The HPV vaccine has been given to millions of girls worldwide for over a decade. No evidence links it to infertility. Cervical cancer is preventable with HPV vaccination.

**Source:** Multiple studies on HPV vaccine safety and effectiveness

**Share this correction:**
Hi! This is false. The HPV vaccine does not cause infertility. It protects against cervical cancer. Please share this with other women.



weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca722-992d-7481-b6bc-f90d0b6a93f6
weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca722-edb2-739d-9e88-1f96b133deaa


[Run 2] Evidence: 2 papers
[Run 2] Response:
**Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

**Share this correction:**
Hi! This is false. HPV vaccine has been given to millions of women worldwide without evidence of causing infertility. It protects against cervical cancer, which kills thousands of women each year.

**Source:** PubMed PMID:27549328, PMID:41497052

Claim: Eating papaya causes miscarriage. Warn all pregnant women!!



...

[Run 1] Evidence: 0 papers
[Run 1] Response:
**Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

There is no evidence that eating papaya during pregnancy causes miscarriage. This is a cultural myth without scientific basis. Papaya is safe and nutritious during pregnancy.

**Source:** NIH — Papaya Safety During Pregnancy

**Share this correction:**
Hi! This is a myth. Eating papaya during pregnancy is safe and healthy.



weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca723-8b8a-7f45-8f70-38ec643798d9


[Run 2] Evidence: 0 papers
[Run 2] Response:
**Verdict: Partially True**
**Confidence: High** | **Evidence: A** | **Severity: Medium**

Unripe papaya contains a latex that may stimulate uterine contractions. However, ripe papaya is safe. The claim is misleading because it does not specify the type of papaya.

**Source:** NIH PubMed — Papaya and Pregnancy

**Share this correction:**
Ripe papaya is safe during pregnancy. Unripe papaya may cause contractions. Always eat ripe fruit.

Claim: Birth control pills cause permanent infertility after 2 years of use.



weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca723-c3a6-7024-9ed3-58a162fe5e37


[Run 1] Evidence: 3 papers
[Run 1] Response:
**Verdict: False**
**Confidence: High** | **Evidence: A**
**Source:** PubMed PMID:8324613

The evidence shows that the majority of women who stop the pill recover fertility within 6 years. The pill does not cause permanent infertility.

[Run 2] Evidence: 3 papers
[Run 2] Response:
**Verdict: ** **Partially True**
**Confidence: ** **Medium** | **Evidence: ** **A**
**Explanation:** Birth control pills do not cause permanent infertility in most women. However, some women may experience temporary amenorrhea (absence of periods) after stopping the pill, which can last up to 6 months in some cases. A small percentage of women may have longer-term effects.
**Source:** Contraception and women's health
**Share this correction:** **Verdict:** **Partially True**. Birth control pills do not cause permanent infertility in most women. Some women may experience temporary amenorrhea after stopping the pill.


In [37]:
# Before/After Eval — Base Mistral 7B vs Fine-tuned
# Shows fine-tuning is necessary — base model can't produce structured verdicts

@weave.op()
def base_model_inference(claim: str) -> str:
    """Base Mistral 7B with NO fine-tuning — just a prompt."""
    messages = [
        {"role": "user", "content": (
            f"Is this women's health claim true or false? Give a verdict.\n\n"
            f"Claim: {claim}"
        )}
    ]
    # Use model but strip LoRA — load base weights only
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(
        input_ids      = inputs,
        max_new_tokens = 300,
        temperature    = 0.3,
        do_sample      = True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

@weave.op()
def finetuned_inference(claim: str) -> str:
    """Fine-tuned Mistral 7B with RAG context."""
    result = full_pipeline_run1(claim)
    return result["response"]

print("BASE MODEL vs FINE-TUNED COMPARISON\n")

for claim in test_claims:
    print(f"\n{'='*60}")
    print(f"Claim: {claim}\n")

    base = base_model_inference(claim)
    print(f"[BASE MODEL]:\n{base}\n")

    ft = finetuned_inference(claim)
    print(f"[FINE-TUNED + RAG]:\n{ft}")

BASE MODEL vs FINE-TUNED COMPARISON


Claim: HPV vaccine causes infertility!! Share with all women!!!



weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca72b-dbf7-75ad-ba74-beffc9eb0bf3


[BASE MODEL]:
**Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

HPV vaccines do not cause infertility. The vaccine stimulates the immune system to fight HPV, but it does not affect fertility. This claim has been studied extensively and found to be untrue.

**Source:** WHO HPV Vaccine Safety Information

**Share this correction:**
Hi! This is false. HPV vaccines do not cause infertility. They protect against HPV, which can cause cervical cancer. Please share accurate information.



weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca72c-02ef-7adf-b45e-48ee91810a35
weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca72c-36af-738d-a5c8-ebca19119e78


[FINE-TUNED + RAG]:
**Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

HPV vaccination has been given to millions of adolescents worldwide without evidence of infertility. HPV vaccination prevents cervical cancer, which is the second most common cancer in women. Refusing vaccination carries far greater risk than any theoretical benefit of avoiding it.

**Source:** PubMed PMID:27549328

**Share this correction:**
Hi! This is false. HPV vaccine does not cause infertility. It protects against cervical cancer. Please vaccinate your daughter if offered.

Claim: Eating papaya causes miscarriage. Warn all pregnant women!!



...

[BASE MODEL]:
**Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: Medium**

Papaya is safe during pregnancy. The claim that it causes miscarriage is not supported by evidence. Green, unripe papaya contains latex that can stimulate uterine contractions in high amounts, but ripe papaya is safe and nutritious.

**Source:** NIH — Papaya and Pregnancy

**Share this correction:**
Hi! This is false. Papaya is safe during pregnancy. Only very large amounts of unripe papaya latex can stimulate contractions. Enjoy ripe papaya in your diet.



weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca72c-8bc8-7250-b442-0958799ed97d


[FINE-TUNED + RAG]:
**Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Papaya does not cause miscarriage. The unripe fruit contains a latex that may stimulate uterine contractions in large amounts, but ripe papaya is safe. This claim has been studied and disproved.

**Source:** NIH — Papaya Safety During Pregnancy

**Share this correction:**
Hi! This is false. Ripe papaya is safe during pregnancy. Only unripe papaya in large amounts may cause mild uterine contractions.

Claim: Birth control pills cause permanent infertility after 2 years of use.



weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca72c-ae39-7a2c-8eb6-aa6fab6364d5


[BASE MODEL]:
**Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: Medium**

Birth control pills do not cause permanent infertility. Fertility returns within 3–6 months of stopping. The pill does not affect the ovaries or uterus in a way that causes permanent damage.

**Source:** ACOG Contraceptive Use and Fertility

**Share this correction:**
Hi! This is false. Birth control pills do not cause permanent infertility. Fertility returns within a few months of stopping.

[FINE-TUNED + RAG]:
**Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: Low**

The evidence shows that the vast majority of women regain their periods after stopping the pill. Only a small percentage of women experience permanent amenorrhea. This is not a permanent effect.

**Source:** PubMed PMID:8324613

**Share this correction:**
Hi! This is false. Most women regain their periods after stopping the pill. Only a small percentage experience permanent changes.


In [2]:
 !git clone --branch elena_mistral_hack https://github.com/elenaajayi/femmestral.git
 !git -C femmestral pull origin elena_mistral_hack

Cloning into 'femmestral'...
remote: Enumerating objects: 241, done.
remote: Counting objects: 100% (241/241), done.
remote: Compressing objects: 100% (210/210), done.
remote: Total 241 (delta 47), reused 214 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (241/241), 452.14 KiB | 5.80 MiB/s, done.
Resolving deltas: 100% (47/47), done.
From https://github.com/elenaajayi/femmestral
 * branch            elena_mistral_hack -> FETCH_HEAD
Already up to date.


In [5]:

# Cell 16 — TRUE base model (no LoRA) vs fine-tuned — GPU-adaptive
# Works on T4, A100, H100, or any CUDA GPU — no hardcoded device.
# If you have H100 credits: Runtime → Change runtime type → H100 before running.

%%capture
!pip install unsloth wandb weave peft

import os, torch, wandb, weave
from google.colab import userdata
from unsloth import FastLanguageModel, is_bfloat16_supported
from peft import PeftModel

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["HF_TOKEN"]      = userdata.get("HF_TOKEN")

weave.init("femmestral")

# Auto-detect GPU and optimal dtype
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
dtype    = torch.bfloat16 if is_bfloat16_supported() else torch.float16
print(f"GPU: {gpu_name} | dtype: {dtype}")

HF_REPO_FT = "elenaajayi/femmestral-mistral-7b-v2"

# --- True base model: NO LoRA ---
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length= 2048,
    dtype         = dtype,
    load_in_4bit  = True,
)
FastLanguageModel.for_inference(base_model)

# --- Fine-tuned model: load base then apply LoRA adapter ---
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length= 2048,
    dtype         = dtype,
    load_in_4bit  = True,
)
ft_model = PeftModel.from_pretrained(ft_model, HF_REPO_FT, token=os.environ["HF_TOKEN"])
FastLanguageModel.for_inference(ft_model)

print("Base and fine-tuned models loaded.")









In [6]:

# Cell 17 — Before/after comparison with Weave tracing

SYSTEM_PROMPT = """You are Femmestral, a women's health misinformation fact-checker.
You will be given a health claim and relevant medical evidence from PubMed.
Respond ONLY in this exact format:

**Verdict: [False/Misleading/Partially True/True]**
**Confidence: [High/Medium/Low]** | **Evidence: [A/B/C]** | **Severity: [High/Medium/Low]**

[2-3 sentence explanation grounded in the provided evidence]

**Source:** [cite the most relevant source]

**Share this correction:**
[1-2 sentence WhatsApp-friendly correction]"""


def _run_model(model, tokenizer, claim: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Claim to fact-check: {claim}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    attention_mask = (inputs != tokenizer.pad_token_id).long()
    outputs = model.generate(
        input_ids      = inputs,
        attention_mask = attention_mask,
        max_new_tokens = 600,
        temperature    = 0.3,
        do_sample      = True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)


@weave.op()
def base_model_inference(claim: str) -> str:
    return _run_model(base_model, base_tokenizer, claim)


@weave.op()
def finetuned_model_inference(claim: str) -> str:
    return _run_model(ft_model, ft_tokenizer, claim)


test_claims = [
    "HPV vaccine causes infertility!! Share with all women!!!",
    "Eating papaya causes miscarriage. Warn all pregnant women!!",
    "Birth control pills cause permanent infertility after 2 years of use.",
    "Mammograms cause breast cancer from the radiation.",
    "Women cannot get pregnant while breastfeeding — it is 100% safe.",
]

rows = []
for claim in test_claims:
    base_out = base_model_inference(claim)
    ft_out   = finetuned_model_inference(claim)
    rows.append([claim, base_out[:400], ft_out[:400]])
    print(f"\nClaim: {claim}")
    print(f"  BASE:        {base_out[:200]}...")
    print(f"  FINE-TUNED:  {ft_out[:200]}...")
    print("=" * 70)

# Log comparison table to W&B
run_cmp = wandb.init(project="femmestral", job_type="eval", name="before-after-comparison")
run_cmp.log({"before_after": wandb.Table(
    columns=["claim", "base_model", "fine_tuned"],
    data=rows
)})
wandb.finish()
print("Comparison logged to W&B.")



Claim: HPV vaccine causes infertility!! Share with all women!!!
  BASE:        **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

The claim that HPV vaccine causes infertility is not supported by scientific evidence. A systematic review published in ...
  FINE-TUNED:  **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

The HPV vaccine has been given to millions of women worldwide with no evidence of fertility effects. The vaccine protect...

Claim: Eating papaya causes miscarriage. Warn all pregnant women!!
  BASE:        **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

There is no scientific evidence to support the claim that eating papaya causes miscarriage. A study published in the Jou...
  FINE-TUNED:  **Verdict: Partially True**
**Confidence: Medium** | **Evidence: A** | **Severity: Medium**

Unripe papaya contains latex that may stimulate uterine contractions and theoretically cause miscarria

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: elenaajayi (elenaajayi-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Initializing weave.


Output()

Comparison logged to W&B.


In [17]:
# Cell 18 — Evaluation (manual scoring + W&B table + Weave traces)
import re, wandb

eval_examples = [
    {"claim": "HPV vaccine causes infertility\!\! Share with all women\!\!\!",      "expected": "False"},
    {"claim": "Eating papaya causes miscarriage. Warn all pregnant women\!\!",   "expected": "False"},
    {"claim": "Birth control pills cause permanent infertility after 2 years.","expected": "False"},
    {"claim": "Mammograms cause breast cancer from the radiation.",             "expected": "False"},
    {"claim": "Women cannot get pregnant while breastfeeding — 100% safe.",    "expected": "Misleading"},
    {"claim": "Ibuprofen should be avoided during early pregnancy.",            "expected": "True"},
    {"claim": "Folic acid prevents neural tube defects.",                       "expected": "True"},
    {"claim": "Women over 40 cannot get pregnant naturally.",                   "expected": "False"},
]

rows = []
correct = 0
format_ok = 0
required_fields = ["**Verdict:", "**Confidence:", "**Evidence:",
                   "**Severity:", "**Source:", "**Share this correction:"]

for ex in eval_examples:
    raw = finetuned_model_inference(ex["claim"])
    m = re.search(r"\*\*Verdict:\s*([^*]+)\*\*", raw)
    predicted = m.group(1).strip() if m else "Unknown"
    verdict_match = int(predicted.lower() == ex["expected"].lower())
    fmt = int(all(r in raw for r in required_fields))
    correct   += verdict_match
    format_ok += fmt
    rows.append([ex["claim"], ex["expected"], predicted, verdict_match, fmt])
    print(f"Claim:     {ex['claim'][:60]}")
    print(f"Expected:  {ex['expected']}  |  Predicted: {predicted}  |  Match: {bool(verdict_match)}")
    print()

n = len(eval_examples)
print(f"Verdict accuracy:  {correct}/{n} = {correct/n:.0%}")
print(f"Format compliance: {format_ok}/{n} = {format_ok/n:.0%}")

run_eval = wandb.init(project="femmestral", job_type="eval", name="femmestral-eval")
run_eval.log({
    "verdict_accuracy":  correct / n,
    "format_compliance": format_ok / n,
    "eval_table": wandb.Table(
        columns=["claim", "expected", "predicted", "verdict_correct", "format_ok"],
        data=rows
    )
})
wandb.finish()
print("Eval logged to W&B.")


Weave Evaluation complete:
{'verdict_correct': None, 'format_correct': None, 'model_latency': {'mean': 0.024153470993041992}}
